# Chat companion prompt eval

Runs the Phase 1 tagging prompt (`backend.chat.TAG_SYSTEM_PROMPT` / `TAG_SCHEMA`) against the labeled cases in `eval/chat_eval_cases.json`, plus a few prose-reply tone checks. Outputs are saved in this notebook on run, so results can be reviewed later without re-calling the API.

In [1]:
import json
import os
import sys
from pathlib import Path

# Make cwd the project root regardless of where this notebook is launched from,
# since st.secrets resolves .streamlit/secrets.toml relative to cwd.
if not (Path.cwd() / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

from backend.chat import CHAT_MODEL, TAG_SCHEMA, TAG_SYSTEM_PROMPT, build_system_prompt
from backend.claude_client import TAG_MODEL, call_prose, call_structured

CASES_PATH = Path.cwd() / "eval" / "chat_eval_cases.json"
cases = json.loads(CASES_PATH.read_text())
len(cases)

12

In [2]:
results = []
for i, case in enumerate(cases, start=1):
    messages = [*case["context"], {"role": "user", "content": case["message"]}]
    tags = call_structured(
        model=TAG_MODEL,
        system=TAG_SYSTEM_PROMPT,
        messages=messages,
        tool_name="tag_message",
        tool_description="Classify the sentiment and repetition of the latest message.",
        tool_schema=TAG_SCHEMA,
    )
    ok = (
        tags["sentiment"] == case["expected_sentiment"]
        and tags["repeated_question_flag"] == case["expected_repeated_question_flag"]
    )
    results.append(
        {
            "case": i,
            "message": case["message"],
            "expected_sentiment": case["expected_sentiment"],
            "got_sentiment": tags["sentiment"],
            "expected_repeated": case["expected_repeated_question_flag"],
            "got_repeated": tags["repeated_question_flag"],
            "pass": ok,
        }
    )

passed = sum(r["pass"] for r in results)
print(f"{passed}/{len(results)} passed\n")
for r in results:
    status = "PASS" if r["pass"] else "FAIL"
    print(f"[{status}] case {r['case']}: {r['message']!r}")
    print(
        f"         sentiment: got={r['got_sentiment']!r} expected={r['expected_sentiment']!r} | "
        f"repeated: got={r['got_repeated']!r} expected={r['expected_repeated']!r}"
    )

2026-07-29 22:07:53.173 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


12/12 passed

[PASS] case 1: 'I feel wonderful today, the weather is lovely!'
         sentiment: got='positive' expected='positive' | repeated: got=False expected=False
[PASS] case 2: 'Just checking in, nothing much happening today.'
         sentiment: got='neutral' expected='neutral' | repeated: got=False expected=False
[PASS] case 3: "I'm a bit tired today, didn't sleep well."
         sentiment: got='low' expected='low' | repeated: got=False expected=False
[PASS] case 4: "I don't see the point anymore, I feel so alone and hopeless."
         sentiment: got='distress' expected='distress' | repeated: got=False expected=False
[PASS] case 5: 'Did I take my medication today?'
         sentiment: got='neutral' expected='neutral' | repeated: got=True expected=True
[PASS] case 6: "What day is my doctor's appointment?"
         sentiment: got='neutral' expected='neutral' | repeated: got=False expected=False
[PASS] case 7: "I'm scared, someone called saying I owe money and I don't know what

## Prose reply tone check

Not a pass/fail eval — just eyeballing what the companion actually sounds like for a couple of representative messages, since tone matters as much as correctness here.

In [3]:
tone_check_cases = [
    ("I've been feeling a bit lonely lately, nobody visits much anymore.", "English"),
    ("Someone called saying my bank account will be frozen unless I pay them today.", "English"),
    ("I've been feeling a bit lonely lately, nobody visits much anymore.", "Mandarin Chinese"),
    ("What's the weather like where you are?", "Malay"),
]

for msg, target_language in tone_check_cases:
    reply = call_prose(
        model=CHAT_MODEL,
        system=build_system_prompt(target_language),
        messages=[{"role": "user", "content": msg}],
    )
    print(f"> [{target_language}] {msg}\n{reply}\n{'-' * 60}")

> [English] I've been feeling a bit lonely lately, nobody visits much anymore.
I'm really sorry you're feeling that way. Loneliness can be so heavy, and I'm glad you shared it with me. 💛

I'm here to chat anytime you'd like company. Is there someone — family, a friend, or neighbor — you could reach out to, even just for a short call? Sometimes a small hello can mean a lot, for you and them too.

What usually lifts your spirits a little?
------------------------------------------------------------


> [English] Someone called saying my bank account will be frozen unless I pay them today.
That sounds scary, but please don't pay anything yet. Banks don't usually call and demand instant payment like that — it's a common scam trick.

A few gentle suggestions:
- Don't give them any money, card details, or codes.
- Hang up if you haven't already.
- Call your bank directly using the number on your card or statement — not the number they gave you.
- Maybe ask a family member to help you check if it's real.

You're doing the right thing by pausing to ask. 💛
------------------------------------------------------------


> [Mandarin Chinese] I've been feeling a bit lonely lately, nobody visits much anymore.
我明白，一个人的时候确实会觉得孤单。您并不孤单，我在这里陪着您呢。

平时您可以多和家人打打电话，哪怕聊几句也好，让他们知道您想他们了。也可以试试联系老朋友，或者参加社区里的小活动，认识新朋友。

如果心里觉得闷，随时都可以跟我说说话，我很愿意听您聊聊天。您今天过得怎么样呀？
------------------------------------------------------------


> [Malay] What's the weather like where you are?
Maaf, saya AI dan tidak berada di mana-mana secara fizikal, jadi saya tak dapat rasa cuaca. 😊

Kalau makcik/pakcik nak tahu cuaca hari ini, boleh tengok di aplikasi cuaca telefon atau tanya ahli keluarga untuk bantu semak. Ada apa-apa lagi saya boleh bantu?
------------------------------------------------------------
